<a href="https://colab.research.google.com/github/avocado-planet/00-LCEL/blob/main/00_LCEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LCEL（LangChain Expression Language）完全ガイド

LCELの全パターンを体系的に学ぶためのノートブックです。  
各セクションは独立して実行可能です。

## 0. セットアップ

In [ ]:
!pip install -q langchain-openai langchain-core langsmith

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 全セクション共通のモデル
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

---
## 1. 基本：パイプ演算子（`|`）によるチェーン

LCELの核心は `|` でRunnableを直列に連結すること。  
前のRunnableの出力が次のRunnableの入力になる。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "与えられた単語の意味を一文で簡潔に説明してください。"),
    ("human", "{word}")
])

# Prompt → Model → OutputParser の基本チェーン
chain = prompt | model | StrOutputParser()

result = chain.invoke({"word": "LCEL"})
print(result)

In [ ]:
# パイプを使わない場合の等価コード（比較用）
prompt_value = prompt.invoke({"word": "LCEL"})
ai_message = model.invoke(prompt_value)
result = StrOutputParser().invoke(ai_message)
print(result)

---
## 2. 呼び出し方法：invoke / stream / batch / ainvoke

すべてのLCELチェーンは統一インターフェースを持つ。

In [ ]:
chain = prompt | model | StrOutputParser()

# --- invoke: 単一入力 → 単一出力 ---
print("=== invoke ===")
print(chain.invoke({"word": "Python"}))
print()

In [ ]:
# --- stream: トークン単位でストリーミング出力 ---
print("=== stream ===")
for chunk in chain.stream({"word": "Python"}):
    print(chunk, end="", flush=True)
print()

In [ ]:
# --- batch: 複数入力を並列処理 ---
print("=== batch ===")
results = chain.batch(
    [{"word": "Python"}, {"word": "Rust"}, {"word": "Go"}],
    config={"max_concurrency": 2}  # 同時実行数を制限可能
)
for r in results:
    print(f"- {r}")

In [ ]:
# --- ainvoke: 非同期版（Colab/Jupyterでは await で呼び出し） ---
print("=== ainvoke ===")
result = await chain.ainvoke({"word": "Python"})
print(result)

---
## 3. RunnableLambda：任意の関数をチェーンに組み込む

Python関数を `RunnableLambda` でラップするか、  
パイプ演算子に直接渡す（自動変換される）。

In [ ]:
from langchain_core.runnables import RunnableLambda

# 方法1: パイプに直接関数を渡す（自動的にRunnableLambdaになる）
def add_exclamation(text: str) -> str:
    return text + "！！！"

chain = prompt | model | StrOutputParser() | add_exclamation
print(chain.invoke({"word": "AI"}))
print()

In [ ]:
# 方法2: 明示的にRunnableLambdaを使う
def word_count(text: str) -> dict:
    """テキストの文字数と単語数を返す"""
    return {
        "text": text,
        "char_count": len(text),
        "word_count": len(text.split())
    }

chain = prompt | model | StrOutputParser() | RunnableLambda(word_count)
result = chain.invoke({"word": "Docker"})
print(result)

In [ ]:
# 方法3: @chain デコレータ（関数をそのままRunnableにする）
from langchain_core.runnables import chain as chain_decorator

@chain_decorator
def analyze_word(input_dict: dict) -> str:
    """単語を受け取り、定義を取得して文字数を付加する"""
    word = input_dict["word"]
    definition = (prompt | model | StrOutputParser()).invoke({"word": word})
    return f"[{word}]（{len(definition)}文字）: {definition}"

print(analyze_word.invoke({"word": "Kubernetes"}))

---
## 4. RunnableParallel：並列実行

複数のRunnableを同時に実行し、結果を辞書にまとめる。  
`{}` 記法でも `RunnableParallel()` でも書ける。

In [ ]:
from langchain_core.runnables import RunnableParallel

# 同じトピックについて異なる視点で並列生成
positive_prompt = ChatPromptTemplate.from_messages([
    ("system", "与えられたテーマのメリットを3つ、箇条書きで述べてください。"),
    ("human", "{topic}")
])

negative_prompt = ChatPromptTemplate.from_messages([
    ("system", "与えられたテーマのデメリットを3つ、箇条書きで述べてください。"),
    ("human", "{topic}")
])

parser = StrOutputParser()

parallel_chain = RunnableParallel(
    merits=positive_prompt | model | parser,
    demerits=negative_prompt | model | parser,
)

result = parallel_chain.invoke({"topic": "リモートワーク"})
print("【メリット】")
print(result["merits"])
print("\n【デメリット】")
print(result["demerits"])

In [ ]:
# 並列結果を後続チェーンに渡す（itemgetterでキーを引き継ぐ）
from operator import itemgetter

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "メリットとデメリットを踏まえて、{topic}について総合的な結論を一段落で述べてください。"),
    ("human", "メリット:\n{merits}\n\nデメリット:\n{demerits}")
])

full_chain = (
    {
        "merits": positive_prompt | model | parser,
        "demerits": negative_prompt | model | parser,
        "topic": itemgetter("topic"),
    }
    | summary_prompt
    | model
    | parser
)

print(full_chain.invoke({"topic": "リモートワーク"}))

---
## 5. RunnablePassthrough：入力をそのまま通す

入力データをそのまま次のステップに渡す。  
RAGパターンなどで、検索結果と元の質問を両方渡したいときに活躍する。

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# 疑似的なretriever（実際はベクトルDBなどを使う）
def fake_retriever(query: str) -> str:
    """検索を模擬する関数"""
    knowledge = {
        "LCEL": "LCELはLangChain Expression Languageの略で、Runnableインターフェースを使ったチェーン構築記法である。",
        "RAG": "RAGはRetrieval-Augmented Generationの略で、検索した文書をLLMのコンテキストに含める手法である。",
    }
    for key, value in knowledge.items():
        if key.lower() in query.lower():
            return value
    return "該当する情報は見つかりませんでした。"

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "以下のコンテキストに基づいて回答してください。\nコンテキスト: {context}"),
    ("human", "{question}")
])

# RunnablePassthrough で question をそのまま通し、context は retriever で取得
rag_chain = (
    {
        "context": itemgetter("question") | RunnableLambda(fake_retriever),
        "question": itemgetter("question"),  # そのまま通す
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

print(rag_chain.invoke({"question": "LCELとは何ですか？"}))

In [ ]:
# RunnablePassthrough.assign() で既存の入力に新しいキーを追加

chain_with_assign = RunnablePassthrough.assign(
    context=lambda x: fake_retriever(x["question"])
) | rag_prompt | model | StrOutputParser()

# questionはそのまま通り、contextが追加される
print(chain_with_assign.invoke({"question": "RAGとは何ですか？"}))

---
## 6. RunnableBranch / router：条件分岐

入力の内容に応じて異なるチェーンを実行する。

In [ ]:
from langchain_core.runnables import RunnableBranch

# カテゴリ別に異なるプロンプトを使う
tech_prompt = ChatPromptTemplate.from_messages([
    ("system", "あなたはITエンジニアです。技術的に正確に回答してください。"),
    ("human", "{question}")
])

cooking_prompt = ChatPromptTemplate.from_messages([
    ("system", "あなたは料理研究家です。家庭でも作りやすいように回答してください。"),
    ("human", "{question}")
])

general_prompt = ChatPromptTemplate.from_messages([
    ("system", "わかりやすく回答してください。"),
    ("human", "{question}")
])

# RunnableBranch: (条件, Runnable) のペアを順に評価し、最初にTrueになったものを実行
branch = RunnableBranch(
    (lambda x: x["category"] == "tech",    tech_prompt | model | StrOutputParser()),
    (lambda x: x["category"] == "cooking", cooking_prompt | model | StrOutputParser()),
    general_prompt | model | StrOutputParser(),  # デフォルト（どの条件にも合わない場合）
)

print("=== tech ===")
print(branch.invoke({"category": "tech", "question": "Dockerとは？"}))
print("\n=== cooking ===")
print(branch.invoke({"category": "cooking", "question": "味噌汁の作り方"}))
print("\n=== default ===")
print(branch.invoke({"category": "other", "question": "今日の天気は？"}))

In [ ]:
# カスタムルーター関数による分岐（RunnableLambda + 辞書ルックアップ）

chain_map = {
    "tech": tech_prompt | model | StrOutputParser(),
    "cooking": cooking_prompt | model | StrOutputParser(),
}

def route(input_dict: dict) -> str:
    category = input_dict["category"]
    selected_chain = chain_map.get(category, general_prompt | model | StrOutputParser())
    return selected_chain.invoke(input_dict)

router_chain = RunnableLambda(route)
print(router_chain.invoke({"category": "tech", "question": "APIとは？"}))

---
## 7. bind：モデルにパラメータやツールを固定する

`model.bind()` で呼び出しパラメータを事前に固定できる。  
Function Calling（Tool use）のバインドにもよく使う。

In [ ]:
# temperature や stop を固定する
creative_model = model.bind(temperature=0.9, stop=["\n\n"])

creative_chain = prompt | creative_model | StrOutputParser()
print(creative_chain.invoke({"word": "宇宙"}))

In [ ]:
# ツール（Function Calling）をbindする例
from langchain_core.pydantic_v1 import BaseModel, Field

class WeatherQuery(BaseModel):
    """天気を調べるためのパラメータ"""
    city: str = Field(description="都市名")
    date: str = Field(description="日付（YYYY-MM-DD形式）")

weather_prompt = ChatPromptTemplate.from_messages([
    ("system", "ユーザの質問から天気を調べるのに必要な情報を抽出してください。"),
    ("human", "{question}")
])

# bind_tools でツールをバインド
model_with_tools = model.bind_tools([WeatherQuery])

tool_chain = weather_prompt | model_with_tools
result = tool_chain.invoke({"question": "明日の東京の天気は？"})
print(result.tool_calls)

---
## 8. with_config / configurable：実行時設定の切り替え

同じチェーンでモデルやパラメータを実行時に切り替える。

In [ ]:
from langchain_core.runnables import ConfigurableField

# モデルのtemperatureを実行時に変更可能にする
configurable_model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0).configurable_fields(
    temperature=ConfigurableField(
        id="llm_temperature",
        name="Temperature",
        description="LLMの出力のランダム性"
    )
)

chain = prompt | configurable_model | StrOutputParser()

# デフォルト（temperature=0）
print("=== temperature=0 ===")
print(chain.invoke({"word": "創造性"}))

# 実行時にtemperatureを変更
print("\n=== temperature=1.0 ===")
print(chain.with_config(configurable={"llm_temperature": 1.0}).invoke({"word": "創造性"}))

In [ ]:
# with_config でメタデータやタグを付与（LangSmithでのトレーシングに有用）
result = chain.with_config(
    run_name="word_definition_chain",
    tags=["demo", "lcel-tutorial"],
    metadata={"user_id": "test-user"}
).invoke({"word": "LangSmith"})

print(result)

---
## 9. チェーンの連結（Chain of Chains）

チェーン同士を `|` で連結。  
前のチェーンの出力型と次のチェーンの入力型が合っていれば繋がる。

In [ ]:
# Chain1: 質問に対してステップバイステップで考える
think_prompt = ChatPromptTemplate.from_messages([
    ("system", "ステップバイステップで考えて回答してください。"),
    ("human", "{question}")
])
think_chain = think_prompt | model | StrOutputParser()

# Chain2: 長い回答を一文に要約する
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "以下の回答から最終的な結論だけを一文で抽出してください。"),
    ("human", "{text}")
])

# StrOutputParser()の出力(str)を summarize_prompt の {text} に渡す
# → RunnableLambda で辞書に変換する必要がある
summarize_chain = (
    RunnableLambda(lambda text: {"text": text})
    | summarize_prompt
    | model
    | StrOutputParser()
)

full_chain = think_chain | summarize_chain

print(full_chain.invoke({"question": "12 * 8 + 15 * 3 の答えは？"}))

---
## 10. OutputParser の種類

LLMの出力を構造化データに変換する。

In [ ]:
# --- StrOutputParser: テキストとして取得 ---
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()
print(type(chain.invoke({"word": "API"})))  # <class 'str'>

In [ ]:
# --- JsonOutputParser: JSON形式で取得 ---
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "与えられた単語について、以下のJSON形式で回答してください。\n"
               '{{"word": "単語", "definition": "定義", "example": "使用例"}}'),
    ("human", "{word}")
])

json_chain = json_prompt | model | JsonOutputParser()
result = json_chain.invoke({"word": "API"})
print(type(result))  # <class 'dict'>
print(result)

In [ ]:
# --- with_structured_output: Pydanticモデルで型安全に取得 ---
from pydantic import BaseModel, Field

class WordInfo(BaseModel):
    word: str = Field(description="単語")
    definition: str = Field(description="定義")
    category: str = Field(description="カテゴリ（技術/科学/日常など）")

structured_model = model.with_structured_output(WordInfo)

structured_chain = prompt | structured_model
result = structured_chain.invoke({"word": "Docker"})
print(type(result))    # <class 'WordInfo'>
print(result.word)     # Docker
print(result.definition)
print(result.category)

---
## 11. RunnableWithMessageHistory：会話履歴の管理

チャットの履歴を自動的に管理し、マルチターン会話を実現する。

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder

# セッション別の履歴ストア
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "あなたは親切なアシスタントです。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

base_chain = chat_prompt | model | StrOutputParser()

chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user-123"}}

print(chain_with_history.invoke({"input": "私の名前はペンです。"}, config=config))
print()
print(chain_with_history.invoke({"input": "私の名前を覚えていますか？"}, config=config))

---
## 12. Fallback：エラー時の代替チェーン

チェーンの実行に失敗した場合、別のチェーンにフォールバックする。

In [ ]:
# わざとエラーを起こすモデル
bad_model = ChatOpenAI(model_name="gpt-nonexistent", max_retries=0)
good_model = ChatOpenAI(model_name="gpt-4o-mini")

# with_fallbacks で代替チェーンを設定
chain_with_fallback = (
    prompt | bad_model | StrOutputParser()
).with_fallbacks(
    [prompt | good_model | StrOutputParser()]
)

# bad_model が失敗 → good_model にフォールバック
print(chain_with_fallback.invoke({"word": "フォールバック"}))

---
## 13. retry / with_retry：リトライ設定

一時的なエラー（レート制限など）に対してリトライを設定する。

In [ ]:
# with_retry でリトライ回数とリトライ対象の例外を指定
import openai

chain_with_retry = (
    prompt
    | model.with_retry(
        stop_after_attempt=3,
        retry_if_exception_type=(openai.RateLimitError,)
    )
    | StrOutputParser()
)

print(chain_with_retry.invoke({"word": "リトライ"}))

---
## 14. pick / assign / pipe：データの変換と受け渡し

辞書のキーの選択・追加・変換を宣言的に行う。

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# --- pick: 辞書から特定のキーだけを取り出す ---
parallel = RunnableParallel(
    upper=RunnableLambda(lambda x: x["text"].upper()),
    lower=RunnableLambda(lambda x: x["text"].lower()),
    original=RunnableLambda(lambda x: x["text"]),
)

# upperだけ取り出す
chain = parallel | RunnableLambda(lambda x: x["upper"])
print(chain.invoke({"text": "Hello World"}))

In [ ]:
# --- assign: 既存の入力に新しいキーを追加 ---

chain = RunnablePassthrough.assign(
    length=lambda x: len(x["text"]),
    upper=lambda x: x["text"].upper(),
)

result = chain.invoke({"text": "hello"})
print(result)
# {'text': 'hello', 'length': 5, 'upper': 'HELLO'}

---
## 15. チェーンの可視化とデバッグ

チェーンの構造を確認する方法。

In [ ]:
chain = prompt | model | StrOutputParser()

# チェーンの入力・出力スキーマを確認
print("=== Input Schema ===")
print(chain.input_schema.model_json_schema())
print("\n=== Output Schema ===")
print(chain.output_schema.model_json_schema())

In [ ]:
# チェーンのグラフ構造を表示
chain.get_graph().print_ascii()

In [ ]:
# 複雑なチェーンのグラフも可視化できる
full_chain.get_graph().print_ascii()

---
## 16. 実践パターン集

よく使われるLCELパターンの実例。

In [ ]:
# パターン1: Map-Reduce（大量テキストの分割処理→統合）

texts = [
    "Pythonは汎用プログラミング言語で、読みやすさが特徴です。",
    "Rustはメモリ安全性と高パフォーマンスを両立する言語です。",
    "JavaScriptはWeb開発の標準的なスクリプト言語です。",
]

# Map: 各テキストのキーワードを抽出
keyword_prompt = ChatPromptTemplate.from_messages([
    ("system", "以下のテキストからキーワードを3つ抽出し、カンマ区切りで出力してください。"),
    ("human", "{text}")
])
keyword_chain = keyword_prompt | model | StrOutputParser()

# Map実行
keywords_list = keyword_chain.batch([{"text": t} for t in texts])
print("=== 各テキストのキーワード ===")
for kw in keywords_list:
    print(f"  {kw}")

# Reduce: キーワードをまとめて総合分析
reduce_prompt = ChatPromptTemplate.from_messages([
    ("system", "以下のキーワード群から、全体のテーマを一文で要約してください。"),
    ("human", "{all_keywords}")
])
reduce_chain = reduce_prompt | model | StrOutputParser()

all_kw = "\n".join(keywords_list)
print("\n=== 総合テーマ ===")
print(reduce_chain.invoke({"all_keywords": all_kw}))

In [ ]:
# パターン2: 自己修正チェーン（生成 → 検証 → 修正）

generate_prompt = ChatPromptTemplate.from_messages([
    ("system", "与えられたお題で俳句を作ってください。必ず五七五の形式で。"),
    ("human", "{topic}")
])

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "以下の俳句が五七五の形式に正しく従っているか確認し、"
               "正しければ'OK'、正しくなければ修正版を出力してください。"),
    ("human", "{haiku}")
])

self_correcting_chain = (
    generate_prompt
    | model
    | StrOutputParser()
    | RunnableLambda(lambda haiku: {"haiku": haiku})
    | review_prompt
    | model
    | StrOutputParser()
)

print(self_correcting_chain.invoke({"topic": "プログラミング"}))

In [ ]:
# パターン3: 多段階パイプライン（翻訳 → 要約 → 感情分析）

translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "以下の日本語テキストを英語に翻訳してください。翻訳文のみ出力。"),
    ("human", "{text}")
])

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the sentiment of the following text. "
               "Output only one word: positive, negative, or neutral."),
    ("human", "{text}")
])

pipeline = (
    translate_prompt | model | StrOutputParser()
    | RunnableLambda(lambda text: {"text": text})
    | RunnablePassthrough.assign(
        sentiment=sentiment_prompt | model | StrOutputParser()
    )
)

result = pipeline.invoke({"text": "今日は天気が良くて、とても楽しい一日でした！"})
print(f"翻訳: {result['text']}")
print(f"感情: {result['sentiment']}")

---
## まとめ：LCELパターン一覧

| # | パターン | 主なクラス/メソッド | 用途 |
|---|---------|-------------------|------|
| 1 | 基本チェーン | `\|` (pipe) | Runnableの直列連結 |
| 2 | 呼び出し | `invoke/stream/batch/ainvoke` | 同期/非同期/ストリーミング/バッチ |
| 3 | 関数組み込み | `RunnableLambda`, `@chain` | 任意のPython関数をチェーンに |
| 4 | 並列実行 | `RunnableParallel`, `{}` 記法 | 複数Runnableを同時実行 |
| 5 | パススルー | `RunnablePassthrough`, `.assign()` | 入力を透過/キー追加 |
| 6 | 条件分岐 | `RunnableBranch` | 条件に応じたルーティング |
| 7 | パラメータ固定 | `.bind()`, `.bind_tools()` | モデル設定やツールの固定 |
| 8 | 設定切替 | `.configurable_fields()`, `with_config()` | 実行時パラメータ変更 |
| 9 | チェーン連結 | chain1 `\|` chain2 | チェーン同士の接続 |
| 10 | 出力解析 | `StrOutputParser`, `JsonOutputParser`, `with_structured_output` | 構造化データ変換 |
| 11 | 会話履歴 | `RunnableWithMessageHistory` | マルチターン会話 |
| 12 | フォールバック | `.with_fallbacks()` | エラー時の代替チェーン |
| 13 | リトライ | `.with_retry()` | 一時エラーの再試行 |
| 14 | データ変換 | `assign`, `pick`, `itemgetter` | 辞書データの操作 |
| 15 | デバッグ | `get_graph()`, `input_schema` | チェーン構造の確認 |